# Stage I. Training the screening gate (ConvNeXt V2-L, 768 x 768)

Stage I is a binary classifier, Grade 0 versus Grades 1-5. Grade 5 (ungradable) is kept on the
positive side so that ungradable images reach the gradability head of Stage III instead of being
cleared as healthy.

**Steps.**
1. Load the development pool and derive binary labels.
2. Split it 80/20, stratified on the six-level grade.
3. Build the data loaders, the model and the loss.
4. Train (resumable at epoch granularity).
5. Inspect the selected checkpoint.

**Outputs** (in `outputs/stage1/convnextv2_768/`): `best_auc_model.pth` (EMA weights of the best
validation-AUC epoch, with its validation threshold), `latest_model.pth`,
`training_history.csv`, `split.csv`.

## 0. Configuration

In [ ]:
from pathlib import Path

# All paths are relative to the repository root (start Jupyter there, or edit them here).
#   data/train/train_grade.csv     development pool labels: image, grade (0-4; 5 = ungradable)
#   data/train/train_image/        development pool images (51,445 in the paper)
#   data/test/test_grade.csv       test pool labels: image, grade (58,689 images)
#   data/test/test_image/          test pool images
DATA_ROOT = Path("data")
TRAIN_CSV = DATA_ROOT / "train" / "train_grade.csv"
TRAIN_IMG_DIR = DATA_ROOT / "train" / "train_image"
TEST_CSV = DATA_ROOT / "test" / "test_grade.csv"
TEST_IMG_DIR = DATA_ROOT / "test" / "test_image"

OUT_ROOT = Path("outputs")
RUN_DIR = OUT_ROOT / "stage1" / "convnextv2_768"     # checkpoints, history, split
EVAL_DIR = OUT_ROOT / "stage1" / "eval"              # prediction caches and reports

PRESET = "convnextv2_768"      # deployed gate; see hierarchiretina/stage1/config.py
NUM_WORKERS = 0                # the original run used 0 workers
RUN_DIR.mkdir(parents=True, exist_ok=True)

import dataclasses
import pandas as pd
import torch
from hierarchiretina.stage1 import data, engine, losses
from hierarchiretina.stage1 import model as M
from hierarchiretina.stage1.config import get_preset

cfg = get_preset(PRESET)
DEVICE = engine.get_device()
pd.Series(dataclasses.asdict(cfg), name=PRESET).to_frame()

## 1. Development pool

Labels: Grade 0 -> No DR, Grades 1-5 -> DR. The paper's pool has 51,445 images
(Grades 0-5: 32,875 / 3,652 / 10,256 / 1,914 / 1,943 / 805).

In [ ]:
df = data.load_development_pool(TRAIN_CSV, TRAIN_IMG_DIR)
n_dr = int(df.binary_label.sum())
print(f"{len(df):,} images | DR {n_dr:,} | No DR {len(df) - n_dr:,}")
df.grade.value_counts().sort_index().rename("images").to_frame()

## 2. Train/validation split

20 % of the pool is held out for validation, stratified on the six-level grade so that the
Grade-1 proportion is the same in both parts (`train_test_split`, seed 42). Expected sizes:
41,156 training and 10,289 validation images.

In [ ]:
train_df, val_df = data.split_development_pool(df, cfg)
print(f"train {len(train_df):,} | val {len(val_df):,}")
pd.concat([train_df.assign(split="train"), val_df.assign(split="val")])[
    ["image", "img_path", "grade", "binary_label", "split"]].to_csv(RUN_DIR / "split.csv",
                                                                   index=False)
pd.DataFrame(data.iter_grade_table(train_df, val_df)).set_index("grade")

## 3. Loaders, model and loss

* Sampler: class-balanced, with Grade-1 images over-sampled 2.5x.
* Model: ConvNeXt V2-L (FCMAE, ImageNet-22k/1k) -> GeM || attention pooling (2 x 1,536) ->
  LayerNorm -> Linear(3072, 1536) -> GELU -> Linear(1536, 1); dropout 0.35; 201,738,243
  parameters.
* Loss: 0.4 BCE (label smoothing 0.03, pos_weight = 1.6 x n_neg/n_pos = 2.833) + 0.6 focal
  (gamma 2, alpha 0.6).

In [ ]:
engine.seed_everything(cfg.seed, cfg.deterministic)
train_loader, val_loader = data.build_loaders(cfg, train_df, val_df, num_workers=NUM_WORKERS)
model = M.build_model(cfg, pretrained=True)
criterion = losses.build_criterion(cfg, train_df.binary_label.values)
print(f"parameters {M.count_parameters(model):,} | pos_weight {criterion.pos_weight.item():.3f}")
print(f"steps/epoch {len(train_loader):,} | effective batch {cfg.effective_batch}")

## 4. Training

AdamW (head 8e-5, backbone 8e-6, weight decay 0.015), 4 warm-up epochs then cosine decay, batch
8 x 6 accumulation steps, EMA 0.9998, mixed precision, gradient clipping at 1.0, at most 45
epochs, early stopping on validation AUC with patience 8. Each epoch the threshold
argmax(2 TPR - FPR), clipped to [0.20, 0.60], is computed on the EMA validation predictions and
stored with the checkpoint. Re-running this cell resumes after the last completed epoch.

In [ ]:
history = engine.fit(cfg, model, train_loader, val_loader, criterion, RUN_DIR, DEVICE,
                     val_grades=val_df.grade.values)
history.tail()

## 5. Selected checkpoint

In the paper run training stopped after epoch 20; the best checkpoint is epoch 12 (validation AUC
0.9546, Grade-1 routing 0.703) and its stored threshold, 0.2391715943813324, is the gate's
operating point.

In [ ]:
meta = M.checkpoint_meta(RUN_DIR / "best_auc_model.pth")
pd.Series({k: meta.get(k) for k in ("epoch", "auc", "grade1_acc", "threshold")})